# 8. RAG 챗봇 품질 대시보드

**질문 처리 현황 → 확인이 필요한 실행 → 대화와 근거 상세** 순서로 살펴봅니다. 고객 평가셋이나 자동 정답 판정은 사용하지 않습니다. `answered`는 답변 완료 상태이며 정답이라는 뜻이 아닙니다.

6·7번의 `chat_sessions`, `chat_messages`, `rag_runs`를 읽습니다. DB 수정·임베딩·LLM 호출은 하지 않습니다. `.venv` 커널에서 위에서부터 실행하세요. UI 구현과 별도로 노트북에서 기록을 확인하는 대시보드입니다.

화면 구성:
1. 기간·버전·테스트 여부 필터
2. 전체 현황과 처리 상태 분포
3. 확인이 필요한 요청 목록
4. 요청 하나의 질문·답변·검색 근거·참고한 대화
5. 버전별 응답 시간·사용 토큰 비교

모든 수치는 필터를 통과한 요청 기준입니다. 데이터가 없으면 0%를 정확도처럼 표시하지 않고 '자료 없음'으로 표시합니다.

In [ ]:
from pathlib import Path
import sys, json, math
from datetime import datetime, timedelta, time, timezone
from zoneinfo import ZoneInfo
from html import escape
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, HTML

project_root = Path('/Users/anna/mle-02-p1-team1')
sys.path.insert(0, str(project_root / 'src')) if str(project_root / 'src') not in sys.path else None
from car_search_rag.common.database_manager import DatabaseManager
load_dotenv(project_root / '.env', override=False)
db = DatabaseManager()
KST = ZoneInfo('Asia/Seoul')

# 이 노트북의 DB 함수는 모두 읽기 전용 트랜잭션을 사용합니다.
def read_rows(query, params=()):
    with db.connect(camel_case_keys=False) as conn:
        conn.execute('SET TRANSACTION READ ONLY')
        return conn.execute(query, params).fetchall()

# 표에 JSON과 긴 문장을 넣을 때 안전하게 문자로 표현합니다.
def plain(value):
    return json.dumps(value, ensure_ascii=False, default=str) if isinstance(value, (dict, list)) else str(value)

def show_text(title, value):
    display(HTML('<h4>' + escape(title) + '</h4><pre style="white-space:pre-wrap;overflow-wrap:anywhere">'
                 + escape(plain(value) if value is not None else '기록 없음') + '</pre>'))

## 1. 조회 조건

종료일도 포함합니다. 예: 10월 1일~5일은 한국 시간 1일 00:00부터 6일 00:00 직전까지입니다. 요청의 시작 시간(created_at)으로 필터합니다.

`record_mode`: all=전체, test=실습 기록만, service=실습 표시가 없는 기록만. 현재 실습은 is_test=True이므로 기본값은 test입니다. pipeline_version과 prompt_hash는 None이면 전체입니다. 변경한 뒤 2번부터 다시 실행하세요.

In [ ]:
start_date = (datetime.now(KST).date() - timedelta(days=30)).isoformat()
end_date = datetime.now(KST).date().isoformat()
record_mode = 'test'
pipeline_version = None
prompt_hash = None

# 실제로 존재하는 버전/프롬프트를 보고 필터 값을 선택합니다.
versions = read_rows("""SELECT DISTINCT config->>'pipeline_version' AS pipeline_version,
    config->>'prompt_hash' AS prompt_hash FROM anna_rag.rag_runs
    ORDER BY pipeline_version, prompt_hash""")
display(pd.DataFrame(versions, columns=['pipeline_version', 'prompt_hash']))

## 2. 요청 목록 읽기

큰 PDF 본문은 요청 상세를 열 때만 가져옵니다. 집계에는 선택 기간 전체의 요청을 사용하며, 목록 표시 개수만 줄입니다. 목록의 상위 100개를 전체 통계인 것처럼 집계하지 않습니다.

In [ ]:
def load_runs(start_date, end_date, record_mode='test', pipeline_version=None, prompt_hash=None):
    if record_mode not in {'all', 'test', 'service'}:
        raise ValueError('record_mode는 all, test, service 중 하나입니다.')
    start = datetime.combine(datetime.fromisoformat(start_date).date(), time.min, KST)
    stop = datetime.combine(datetime.fromisoformat(end_date).date() + timedelta(days=1), time.min, KST)
    if start >= stop:
        raise ValueError('시작일은 종료일보다 늦을 수 없습니다.')
    # SQL 값은 문자열에 직접 붙이지 않고 매개변수로 전달합니다.
    test_filter = None if record_mode == 'all' else record_mode == 'test'
    rows = read_rows("""
        SELECT r.request_id, r.session_id, r.created_at, r.completed_at, r.status,
               r.duration_ms, r.llm_usage, r.error_type, r.search_question,
               u.content AS question, s.is_test,
               r.config->>'pipeline_version' AS pipeline_version,
               r.config->>'prompt_hash' AS prompt_hash,
               r.snapshot->'answer'->>'reason' AS reason,
               r.snapshot->>'agent_error_type' AS agent_error_type,
               r.snapshot->'tool_calls' AS tool_calls
        FROM anna_rag.rag_runs r JOIN anna_rag.chat_sessions s USING(session_id)
        JOIN anna_rag.chat_messages u ON u.message_id=r.user_message_id AND u.session_id=r.session_id
        WHERE r.created_at >= %s AND r.created_at < %s
          AND (%s::boolean IS NULL OR s.is_test=%s)
          AND (%s::text IS NULL OR r.config->>'pipeline_version'=%s)
          AND (%s::text IS NULL OR r.config->>'prompt_hash'=%s)
        ORDER BY r.created_at, r.request_id
    """, (start, stop, test_filter, test_filter, pipeline_version, pipeline_version, prompt_hash, prompt_hash))
    columns = ['request_id','session_id','created_at','completed_at','status','duration_ms','llm_usage',
               'error_type','search_question','question','is_test','pipeline_version','prompt_hash',
               'reason','agent_error_type','tool_calls']
    return pd.DataFrame(rows, columns=columns)


def token_sum(calls, field):
    # 기록이 없거나 한 호출이라도 값이 없으면 합계를 알 수 없는 것으로 남깁니다.
    if not isinstance(calls, list) or not calls:
        return None
    values = [c.get(field) if isinstance(c, dict) else None for c in calls]
    if any(not isinstance(v, (int, float)) or isinstance(v, bool) or v < 0 for v in values):
        return None
    return sum(values)


def prepare_metrics(frame):
    df = frame.copy()
    for name in ('created_at', 'completed_at'):
        df[name] = pd.to_datetime(df[name], utc=True)
    df['completed'] = df['completed_at'].notna()
    df['response_seconds'] = pd.to_numeric(df['duration_ms'], errors='coerce') / 1000
    for field in ('input_tokens', 'output_tokens', 'total_tokens'):
        df[field] = pd.to_numeric(df['llm_usage'].map(lambda calls: token_sum(calls, field)), errors='coerce')
    # 6번 기록에는 도구 항목이 없습니다. 0회라고 가정하지 않고 미기록으로 표시합니다.
    df['tool_count'] = pd.to_numeric(df['tool_calls'].map(lambda x: len(x) if isinstance(x, list) else None), errors='coerce')
    return df

runs = prepare_metrics(load_runs(start_date, end_date, record_mode, pipeline_version, prompt_hash))
print(f'조회 조건: {start_date} ~ {end_date} (한국 시간), {record_mode}')
print('필터를 통과한 요청 수:', len(runs))

## 3. 전체 현황

비율의 분모는 **완료 시간이 기록된 요청**입니다. 인사·되묻기·범위 밖 응답도 완료에 포함합니다. 진행 중인 요청은 별도 표시합니다.

오류율은 error/agent_error/intent_error/invalid_citations/missing_tool/tool_limit 상태를 집계합니다. 근거 없음과 정상적인 되묻기는 오류로 세지 않습니다. 답변 완료율은 answered / 완료 요청이며 **정답률이 아닙니다**. 응답 시간은 기록된 처리 시간(duration_ms) 기준이며 화면 렌더링·최종 DB 저장 시간까지 포함한 사용자 체감 시간이 아닙니다.

In [ ]:
ERROR_STATUSES = {'error', 'agent_error', 'intent_error', 'invalid_citations', 'missing_tool', 'tool_limit'}
NO_EVIDENCE_STATUSES = {'no_results', 'insufficient_evidence'}

def summarize(df):
    done = df.loc[df['completed']]
    n = len(done)
    def rate(mask):
        return f'{mask.sum() / n * 100:.1f}%' if n else '자료 없음'
    times = done['response_seconds'].dropna()
    return pd.DataFrame([
        ('전체 요청', len(df)), ('대화방 수', df['session_id'].nunique()),
        ('완료 요청', n), ('미완료 요청', len(df)-n),
        ('답변 완료율 (정답률 아님)', rate(done['status'].eq('answered'))),
        ('되묻기 비율', rate(done['status'].eq('needs_clarification'))),
        ('오류율', rate(done['status'].isin(ERROR_STATUSES))),
        ('평균 처리 시간', f'{times.mean():.2f}초' if len(times) else '자료 없음'),
        ('P95 처리 시간', f'{times.quantile(.95):.2f}초' if len(times) else '자료 없음'),
        ('시간 기록이 있는 완료 요청', len(times)),
        ('LLM 토큰 기록이 있는 요청', int(df['total_tokens'].notna().sum())),
        ('기록된 LLM 토큰 합계', df['total_tokens'].sum(min_count=1))
    ], columns=['지표', '값'])

display(summarize(runs))
# 상태별 집계와 회색 막대로 분포를 표시합니다. 사용자 텍스트는 HTML escape 처리합니다.
def show_status_chart(df):
    counts = df['status'].fillna('미기록').value_counts()
    if counts.empty:
        print('표시할 실행 기록이 없습니다. 7번에서 대화를 저장한 뒤 다시 조회하세요.')
        return
    display(counts.rename_axis('처리 상태').reset_index(name='요청 수'))
    bars = []
    for status, count in counts.items():
        width = 100 * count / counts.max()
        bars.append(f'<div style="margin:8px 0">{escape(str(status))}: {int(count)}건'
                    f'<div style="height:12px;background:#888;width:{width:.2f}%"></div></div>')
    display(HTML('<div style="max-width:650px">' + ''.join(bars) + '</div>'))
show_status_chart(runs)

## 4. 확인이 필요한 요청

되묻기/근거 없음은 반드시 실패라는 뜻은 아닙니다. 정상 동작인지 살펴볼 대상으로 모읍니다. 오래 미완료인 요청도 자동 실패로 단정하지 않습니다. 상태를 덮어쓰거나 재실행하지 않고 조회만 합니다.

In [ ]:
now_utc = pd.Timestamp.now(tz='UTC')
age_minutes = (now_utc - runs['created_at']).dt.total_seconds() / 60
runs['long_running'] = ~runs['completed'] & age_minutes.gt(5)
attention_statuses = ERROR_STATUSES | NO_EVIDENCE_STATUSES | {'needs_clarification', 'needs_scope_review', 'context_incomplete', 'input_too_long'}
attention = runs.loc[runs['status'].isin(attention_statuses) | runs['long_running']].copy()
print('확인이 필요한 요청 전체:', len(attention), '/ 아래 표는 최근 100건')
with pd.option_context('display.max_colwidth', 100):
    display(attention.tail(100)[['request_id', 'question', 'status', 'reason', 'long_running', 'response_seconds']])
print('의도/예산 등 사유별 건수:')
display(attention['reason'].fillna('별도 사유 미기록').value_counts().rename_axis('사유').reset_index(name='건수'))

## 5. 질문 하나의 대화·검색 근거 상세

아래 request_id를 문제 목록에서 복사하거나, 그대로 두면 현재 필터의 최신 요청을 봅니다. 필터 밖 요청은 표시하지 않습니다.

실행 당시 snapshot에 저장한 본문을 보여줍니다. 현재 PDF/DB의 본문이 나중에 바뀌어도 과거에 어떤 자료를 전달했는지 확인할 수 있습니다. 검색 점수(distance)는 정답 확률이 아닙니다. 출처 라벨 검사 역시 의미상 정확도 검사가 아닙니다.

In [ ]:
def show_request(request_id, filtered_runs):
    if str(request_id) not in set(filtered_runs['request_id'].astype(str)):
        raise ValueError('현재 조회 조건 안의 request_id를 선택하세요.')
    rows = read_rows("""SELECT r.*, u.content AS original_question, u.turn_no,
        a.content AS answer_text FROM anna_rag.rag_runs r
        JOIN anna_rag.chat_messages u ON u.message_id=r.user_message_id AND u.session_id=r.session_id
        LEFT JOIN anna_rag.chat_messages a ON a.message_id=r.assistant_message_id AND a.session_id=r.session_id
        WHERE r.request_id=%s""", (request_id,))
    if not rows:
        print('요청 기록이 없습니다. 목록을 새로 조회하세요.'); return
    row = rows[0]
    snapshot = row['snapshot'] or {}
    answer = snapshot.get('answer', {})
    show_text('원래 질문', row['original_question'])
    show_text('맥락을 반영한 검색 질문', row['search_question'])
    show_text('챗봇 답변', row['answer_text'])
    show_text('상태 / 사유', {'status': row['status'], 'reason': answer.get('reason'),
                            'error_type': row['error_type'], 'agent_error_type': snapshot.get('agent_error_type')})
    sources = snapshot.get('sources', [])
    cited = set(answer.get('cited_labels', []))
    source_frame = pd.DataFrame([{**s, '답변에서 인용': s.get('label') in cited} for s in sources])
    print('검색된 출처 (인용하지 않은 후보도 포함):')
    display(source_frame)
    show_text('검색 청크 원문', snapshot.get('evidence'))
    show_text('부모 본문', snapshot.get('parent_context'))
    show_text('제외된 부모', snapshot.get('omitted_parents'))
    print('도구 실행 기록:')
    display(pd.DataFrame(snapshot.get('tool_calls', [])))
    # 참고했던 메시지 ID만 현재 세션 안에서 조회합니다. 다른 세션으로 범위를 넓히지 않습니다.
    ids = snapshot.get('history_message_ids', [])
    history = read_rows("""SELECT message_id, turn_no, role, content FROM anna_rag.chat_messages
        WHERE session_id=%s AND message_id::text=ANY(%s::text[])
        ORDER BY turn_no, CASE role WHEN 'user' THEN 0 ELSE 1 END""", (row['session_id'], ids)) if ids else []
    print('실제로 참고한 이전 대화:')
    with pd.option_context('display.max_colwidth', None):
        display(pd.DataFrame(history))
    # 이미지 파일은 자동 다운로드하지 않고 연결 관계와 Storage 경로를 확인합니다.
    print('관련 이미지 후보 (직접 해석/정확성 검증 결과가 아님):')
    display(pd.DataFrame(snapshot.get('image_candidates', [])))
    show_text('기록된 LLM 사용량', row['llm_usage'])
    config = row['config'] or {}
    show_text('실행 설정', {k: v for k, v in config.items() if k != 'prompts'})

selected_request_id = str(runs.iloc[-1]['request_id']) if not runs.empty else None
if selected_request_id:
    show_request(selected_request_id, runs)
else:
    print('조회할 기록이 없습니다.')

## 5-1. 선택 목록으로 상세 보기 (선택)

True로 바꾸면 드롭다운에서 최근 100개 요청 중 하나를 골라 볼 수 있습니다. ipywidgets가 없는 환경에서는 위의 selected_request_id를 직접 바꾸면 됩니다. 각 선택은 DB 조회만 실행합니다.

In [ ]:
SHOW_SELECTOR = False
if SHOW_SELECTOR and not runs.empty:
    try:
        import ipywidgets as widgets
        options = [(f"{str(r['request_id'])[:8]} · {r['question'][:45]}", str(r['request_id']))
                   for _, r in runs.tail(100).iloc[::-1].iterrows()]
        selector = widgets.Dropdown(options=options, description='질문 선택:')
        detail_output = widgets.Output()
        def update_detail(change):
            with detail_output:
                detail_output.clear_output(wait=True)
                show_request(change['new'], runs)
        selector.observe(update_detail, names='value')
        display(selector, detail_output)
        update_detail({'new': selector.value})
    except ImportError:
        print('ipywidgets가 없습니다. 5번의 request_id로 직접 조회하세요.')

## 6. 버전·프롬프트별 비교

같은 버전 이름이어도 프롬프트 해시가 다르면 별도 그룹입니다. 표본 수와 질문 구성이 다를 수 있으므로 단순 수치 차이를 개선 효과나 인과관계로 단정하지 않습니다. 정답률·자동 품질 점수는 계산하지 않습니다.

LLM 토큰은 저장된 호출의 관측값입니다. 누락된 호출이 있을 수 있고 임베딩 토큰은 포함하지 않으므로 전체 사용 비용으로 표시하지 않습니다. 실제 API 가격을 적용하는 비용 계산은 이 노트북 범위에 포함하지 않습니다.

In [ ]:
def compare_versions(df):
    items = []
    for (version, prompt), group in df.groupby(['pipeline_version', 'prompt_hash'], dropna=False):
        done = group.loc[group['completed']]
        latency = done['response_seconds'].dropna()
        items.append({'버전': version, '프롬프트 해시': prompt, '전체 요청': len(group), '완료 요청': len(done),
            '답변 완료율(%)': 100 * done['status'].eq('answered').mean() if len(done) else None,
            '오류율(%)': 100 * done['status'].isin(ERROR_STATUSES).mean() if len(done) else None,
            '평균 시간(초)': latency.mean(), 'P95 시간(초)': latency.quantile(.95),
            '토큰 기록 요청': group['total_tokens'].notna().sum(),
            '관측 토큰 합계': group['total_tokens'].sum(min_count=1),
            '도구 횟수 기록 요청': group['tool_count'].notna().sum(),
            '평균 도구 호출(기록 있는 요청)': group['tool_count'].mean()})
    return pd.DataFrame(items)
display(compare_versions(runs))

## 7. 팀 검토와 다음 단계

자동 통계로 정답 여부를 판단할 수 없습니다. 대화 상세에서 필요한 예시를 골라 아래 질문으로 검토하세요. 고객 평가셋은 만들지 않습니다.

- 검색 자료가 사용자의 질문에 적절한가?
- 답변의 사실이 해당 근거와 일치하는가?
- 조건·경고·금지사항을 빠뜨리지 않았는가?
- 운전자가 이해하기 쉬운가?

이번 노트북은 **조회 전용**이며 팀 검토 입력을 DB에 저장하지 않습니다. 필요하면 이후 review 테이블과 입력 화면을 추가합니다. 다음에는 채팅 UI와 데이터/도구 가이드를 만들고 이 조회 로직을 팀용 대시보드 화면으로 옮길 수 있습니다. 차종 통합은 제외합니다.